<a href="https://colab.research.google.com/github/keerthireddy-23/DataMining-CustomerSegmentation/blob/main/Customer_Segmentaion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("customer segmentation dataset.csv")
print("Original shape: ",df.shape)
df.head()


In [ ]:
df.isnull().sum()

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
for col in num_cols:
  df[col] = df[col].fillna(df[col].median())
for col in cat_cols:
  df[col] = df[col].fillna(df[col].mode()[0])
print("Missing values handled safely")

In [ ]:
df = df.drop_duplicates()
print("After duplicate removal: ", df.shape)

In [ ]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)


print("After outlier removal:", df.shape)

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns

if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    print("Encoding done")
else:
    print("No categorical columns left to encode")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Select only numerical columns
scale_cols = df.select_dtypes(include=['int64', 'float64']).columns

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaling completed")



In [ ]:

df_reduced = df.sample(n=500, random_state=42)

print("Original shape:", df.shape)

In [ ]:

print("feature shape", df.shape)

In [ ]:
print(df.columns)

In [ ]:
X = df[['Annual_Income_k$', 'Spending_Score']]


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
wcss = []

for i in range(2 , 7):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot Elbow Graph
plt.figure(figsize=(8,5))
plt.plot(range(2, 7), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)

clusters = kmeans.fit_predict(X_scaled)

# Add cluster column to dataset
df['Cluster'] = clusters

print(df.head())

In [ ]:
print(df['Cluster'].value_counts())

In [ ]:
cluster_means = df.groupby('Cluster').mean()
print(cluster_means)

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=df['Annual_Income_k$'],
                y=df['Spending_Score'],
                hue=df['Cluster'],
                palette='Set1')

plt.title('Customer Segments')
plt.xlabel('Annual_Income_k$')
plt.ylabel('Spending_Score')
plt.show()

In [ ]:
df.to_csv("Customer_Segmentation_Preprocessed.csv", index=False)

In [ ]:
from google.colab import files
files.download("Customer_Segmentation_Preprocessed.csv")

In [ ]:
# Import 3D plotting toolkit
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

# Create figure
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot
scatter = ax.scatter(
    df['Age'],
    df['Annual_Income_k$'],
    df['Spending_Score'],
    c=df['Cluster'],          # Color based on cluster
    cmap='viridis',
    s=60
)

# Labels
ax.set_xlabel('Age')
ax.set_ylabel('Annual_Income_k$')
ax.set_zlabel('Spending_Score')

# Title
ax.set_title('3D Customer Segmentation')

# Add legend
legend = ax.legend(*scatter.legend_elements(), title="Clusters")
ax.add_artist(legend)

# Show plot
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score
score = silhouette_score(X_scaled, df['Cluster'])
print("Silhouette Score:", score)
